# BRFSS: post-audit development runner

The historical2015 final test is closed to new development. This master runs development-only extensions and displays frozen evidence. Original master is preserved in notebooks/archive. Actual Colab execution remains unverified.

NumPy compatibility: `>=2.0.2,<2.3`. Colab NumPy 2.0.2 is supported without upgrading to 2.2.6. If installation changes an already imported binary package, restart the session and rerun setup. Existing frozen lock files document previous experiments; they are not Colab installation requirements.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("Validated Python range3.10–3.12; revalidate dependencies otherwise.")
SYNTHETIC = os.environ.get("BRFSS_SYNTHETIC", "0") == "1"
if not SYNTHETIC:
    from google.colab import drive

    drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/BRFSS2015_Statistical_Methods_AZ")
REPO = Path(
    os.environ.get(
        "BRFSS_REPO",
        str(Path.cwd()) if SYNTHETIC else str(PROJECT / "BRFSS2015-Statistical-Methods"),
    )
)
if not (REPO / "pyproject.toml").is_file():
    raise FileNotFoundError("Extract the complete research ZIP and set BRFSS_REPO.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
PROFILE = os.environ.get("BRFSS_PROFILE", "SMOKE" if SYNTHETIC else "QUICK")
OUT = Path(os.environ.get("BRFSS_OUT", str(PROJECT / "runs/post_audit_development")))
DATA = os.environ.get("BRFSS_DATA")

In [ ]:
if not SYNTHETIC:
    for requirement in [
        "requirements.txt",
        "requirements-boosters.txt",
        "requirements-research-extensions.txt",
    ]:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", requirement], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
from scripts.verify_environment import verify

# Accepts Colab NumPy 2.0.2 through 2.2.x; checks loaded vs installed versions.
versions = verify()

In [ ]:
from src.data.ingest import discover
import yaml

if not SYNTHETIC:
    cfg = yaml.safe_load(Path("configs/project.yaml").read_text())
    DATA = str(discover(DATA, cfg["candidate_directories"]))
    if not Path("runs/real_final_quick/split.json").is_file():
        raise FileNotFoundError(
            "Use the COMPLETE research ZIP containing the protected legacy run. Do not regenerate a2015 final test."
        )

In [ ]:
from src_extensions.runner import run

run(data=DATA, out=OUT, profile=PROFILE, synthetic=SYNTHETIC)

In [ ]:
from src.utils.io import read_json

integrity = read_json(OUT / "integrity.json")
assert integrity["unchanged"] and integrity["legacy_test_overlap"] == 0
print(integrity["new_final_test_evaluations"], "new2015 final evaluations")
if not SYNTHETIC:
    from src_extensions.ancillary import run as ancillary

    ancillary(OUT)

Temporal2017 evaluation is already frozen/evaluated in the shipped evidence. Do not develop new methods against those results. Read reports/EXTERNAL_VALIDATION_FEASIBILITY.md and UNFINISHED_WORK.md. The current master does not automatically open another test or rerun legacy evaluation.

In [ ]:
from IPython.display import display, Markdown

if Path("reports/FINAL_RESEARCH_REPORT.md").exists():
    display(Markdown(Path("reports/FINAL_RESEARCH_REPORT.md").read_text()))
print("Current execution:", OUT, "profile", PROFILE, "synthetic", SYNTHETIC)
print(
    "Displayed course manuscript is the shipped, dated execution report; it is not automatically relabelled as results of a new run."
)
print("COLAB_RUNTIME_NOT_VERIFIED unless this notebook has actually been executed in Colab.")